In [ ]:
#using Pkg
#Pkg.activate("../../FUSE")
using TurbulentTransport
using Plots
using Measurements

In [ ]:
filename = "sat3_em_d3d_azf-1_withnegD.bson"
fname = split(filename, ".")[1]
model_dir = "../models/$fname"
if !isdir(model_dir)
    mkpath(model_dir)
end

# Extract flux models and xnames
ensemble = TurbulentTransport.loadmodelonce(filename)
fluxmodels = [model.fluxmodel for model in ensemble.models]
xnames = ensemble.models[1].xnames
ynames = ensemble.models[1].ynames
xm = ensemble.models[1].xm
xσ = ensemble.models[1].xσ
ym = ensemble.models[1].ym
yσ = ensemble.models[1].yσ

# Populate xtest dictionary with mean values
xtest = Dict()
for (i, name) in enumerate(xnames)
    clean_name = replace(name, "_log10" => "")
    xtest[clean_name] = [xm[i]]
end

#=xtest["RLNS_1"] = 2.22
xtest["RLTS_1"] = 4.32
xtest["RLTS_2"] = 2.41
xtest["TAUS_2"] = 0.681
xtest["RMIN_LOC"] = 0.82
xtest["Q_LOC"] = 17.9
xtest["KAPPA_LOC"] = 2.48
xtest["DELTA_LOC"] = 0.297
xtest["BETAE"] = log10(0.00407)
xtest["ZEFF"] = 2.11=#

# Prepare the input vector for the model
x = zeros(Float32, length(xnames))
for (i, name) in enumerate(xnames)
    clean_name = replace(name, "_log10" => "")
    if haskey(xtest, clean_name)
        value = xtest[clean_name][1]  # Extract the scalar from the array
        x[i] = Float32(value)
    else
        error("Key '$clean_name' not found in xtest")
    end
end

# Identify relevant indices in xnames
dx1 = Int[]
for i in 1:length(xnames)
    if occursin("RL", xnames[i])
        push!(dx1, i)
    end
end

function delog10(x::Vector, xnames::Vector)
    x = copy(x)
    for ix in findall(map(name -> contains(name, "_log10"), xnames))
        x[ix] = 10.0 .^ (x[ix])
    end
    return x
end

function f(model, x1, xi) 
    model(hcat([delog10(vcat(x[1:xi-1], xx, x[xi+1:end]), xnames) for xx in x1]...); uncertain=true, warn_nn_train_bounds=false)
end

# Loop through all four outputs
for yi in 1:4  # 1: Gamma_e, 2: P_i, 3: Q_e, 4: Q_i
    
    # Initialize a list to store each subplot for this output
    plots = []
    
    for xi in 1:length(xnames)
        # Skip ZMAJ_LOC plots
        if occursin("ZMAJ_LOC", xnames[xi])
            continue
        end
        
        y = ensemble(delog10(x, xnames); warn_nn_train_bounds=false, uncertain=true)

        xrange = LinRange(x[xi] - 3.3 * xσ[xi], x[xi] + 3.3 * xσ[xi], 100)
        
        # Special cases for specific variables
        if xnames[xi] == "Q_LOC"
            xrange = LinRange(max(0.5, x[xi] - 3.3 * xσ[xi]), x[xi] + 3.3 * xσ[xi], 100)
        elseif xnames[xi] == "DRMAJDX_LOC"
            xrange = LinRange(-0.8, 0.1, 100)
        elseif xnames[xi] == "P_PRIME_LOC"
            xrange = LinRange(-0.04, 0.005, 100)
        elseif xnames[xi] == "ZEFF"
            xrange = LinRange(max(1.0, x[xi] - 3.3 * xσ[xi]), x[xi] + 3.3 * xσ[xi], 100)
        elseif xnames[xi] in ["AS_3", "RMIN_LOC"]
            xrange = LinRange(max(0.0, x[xi] - 3.3 * xσ[xi]), x[xi] + 3.3 * xσ[xi], 100)
        end
        
        tmp = f(ensemble, xrange, xi)[yi, :]

        # List of variables that need reduced tick frequency
        reduced_tick_vars = ["AS_3", "BETAE_log10", "DEBYE_log10", "DELTA_LOC", "DRMAJDX_LOC", 
                            "P_PRIME_LOC", "RMAJ_LOC", "RMIN_LOC", "S_DELTA_LOC", "S_ZETA_LOC", 
                            "XNUE_log10", "ZEFF", "ZETA_LOC"]
        
        # Determine tick positions
        if xnames[xi] in reduced_tick_vars
            # Create roughly half the default number of ticks by using fewer points
            n_ticks = 4
            tick_positions = range(minimum(xrange), maximum(xrange), length=n_ticks)
            
            # Format tick labels - special case for DEBYE_log10
            if xnames[xi] == "DEBYE_log10"
                tick_labels = [string(round(tick, sigdigits=2)) for tick in tick_positions]
            else
                tick_labels = [string(round(tick, sigdigits=1)) for tick in tick_positions]
            end
            
            p = plot(xrange, Measurements.value.(tmp), ribbon=Measurements.uncertainty.(tmp),
                    xlabel=xnames[xi], ylabel="", label="", linewidth=2, color=:red,
                    fillalpha=0.5, xtickfontsize=6, ytickfontsize=6, xticks=(tick_positions, tick_labels))
        else
            p = plot(xrange, Measurements.value.(tmp), ribbon=Measurements.uncertainty.(tmp),
                    xlabel=xnames[xi], ylabel="", label="", linewidth=2, color=:red,
                    fillalpha=0.5, xtickfontsize=6, ytickfontsize=6)
        end
        
        scatter!(p, [x[xi]], [y[yi]], color=:blue, label="")
        vline!(p, [xm[xi]], color=:black, linestyle=:dash, label="")
        hline!(p, [0], color=:black, linestyle=:solid, label="")
        
        # Special y-axis limits for specific variables with electron heat flux (Q_elec)
        if yi == 3  # yi=3 is Q_elec (electron heat flux)
            if xnames[xi] in ["BETAE_log10", "P_PRIME_LOC", "Q_LOC", "TAUS_2", "TAUS_3"]
                ylims!(p, 0, 5)
            elseif xnames[xi] == "RLNS_1"
                ylims!(p, -10, 15)
            end
        end

        push!(plots, p)
    end

    # Arrange all plots in a tile layout
    n_cols = 6
    n_rows = cld(length(plots), n_cols)
    plot_grid = plot(plots..., layout=(n_rows, n_cols), size=(1000, 800))

    display(plot_grid)
    savefig(plot_grid, "./plot_spot_check_tglf_$(split(filename,".")[1])_$(ynames[yi]).pdf")
    
    println("Completed TGLF plots for $(ynames[yi])")
end

In [ ]:
filename = "sat3_em_d3d_azf-1_withnegD.bson"
fname = split(filename, ".")[1]
model_dir = "../models/$fname"
if !isdir(model_dir)
    mkpath(model_dir)
end

# Extract flux models and xnames
ensemble = TurbulentTransport.loadmodelonce(filename)
gknn = TurbulentTransport.loadmodelonce("sat3_em_d3d_azf-1_withnegD_gknn31")
fluxmodels = [model.fluxmodel for model in ensemble.models]
xnames = ensemble.models[1].xnames
ynames = ensemble.models[1].ynames
xm = ensemble.models[1].xm
xσ = ensemble.models[1].xσ
ym = ensemble.models[1].ym
yσ = ensemble.models[1].yσ

# Populate xtest dictionary with mean values
xtest = Dict()
for (i, name) in enumerate(xnames)
    clean_name = replace(name, "_log10" => "")
    xtest[clean_name] = [xm[i]]
end

#=xtest["RLNS_1"] = 2.22
xtest["RLTS_1"] = 4.32
xtest["RLTS_2"] = 2.41
xtest["TAUS_2"] = 0.681
xtest["RMIN_LOC"] = 0.82
xtest["Q_LOC"] = 17.9
xtest["KAPPA_LOC"] = 2.48
xtest["DELTA_LOC"] = 0.297
xtest["BETAE"] = log10(0.00407)
xtest["ZEFF"] = 2.11=#

# Prepare the input vector for the model
x = zeros(Float32, length(xnames))
for (i, name) in enumerate(xnames)
    clean_name = replace(name, "_log10" => "")
    if haskey(xtest, clean_name)
        value = xtest[clean_name][1]  # Extract the scalar from the array
        x[i] = Float32(value)
    else
        error("Key '$clean_name' not found in xtest")
    end
end

# Identify relevant indices in xnames
dx1 = Int[]
for i in 1:length(xnames)
    if occursin("RL", xnames[i])
        push!(dx1, i)
    end
end

function delog10(x::Vector, xnames::Vector)
    x = copy(x)
    for ix in findall(map(name -> contains(name, "_log10"), xnames))
        x[ix] = 10.0 .^ (x[ix])
    end
    return x
end

function f(model, x1, xi) 
    model(hcat([delog10(vcat(x[1:xi-1], xx, x[xi+1:end]), xnames) for xx in x1]...); uncertain=true, warn_nn_train_bounds=false)
end

function f_gknn(model, gknn_model, x1, xi)
    # First get TGLFNN predictions
    x_matrix = hcat([delog10(vcat(x[1:xi-1], xx, x[xi+1:end]), xnames) for xx in x1]...)
    y_tglf = model(x_matrix; uncertain=true, warn_nn_train_bounds=false)
    
    # Extract values for GKNN model (it can't handle Measurement objects)
    y_tglf_values = Measurements.value.(y_tglf)
    corrected_inputs = vcat(x_matrix, y_tglf_values)
    err = gknn_model(corrected_inputs; uncertain=false, warn_nn_train_bounds=false, fidelity=:GKNN)
    
    # Apply correction to the original uncertain TGLF predictions
    return y_tglf .* err
end

# Loop through all four outputs
for yi in 1:4  # 1: Gamma_e, 2: P_i, 3: Q_e, 4: Q_i
    
    # Initialize a list to store each subplot for this output
    plots = []
    
    for xi in 1:length(xnames)
        # Skip ZMAJ_LOC plots
        if occursin("ZMAJ_LOC", xnames[xi])
            continue
        end
        
        # Calculate reference points for both models
        y_tglf = ensemble(delog10(x, xnames); warn_nn_train_bounds=false, uncertain=true)
        
        # Calculate GKNN reference point
        x_matrix_ref = reshape(delog10(x, xnames), :, 1)
        y_tglf_values_ref = Measurements.value.(y_tglf)
        corrected_inputs_ref = vcat(x_matrix_ref, y_tglf_values_ref)
        err_ref = gknn(corrected_inputs_ref; uncertain=false, warn_nn_train_bounds=false, fidelity=:GKNN)
        y_gknn = y_tglf .* err_ref

        xrange = LinRange(x[xi] - 3.3 * xσ[xi], x[xi] + 3.3 * xσ[xi], 100)
        
        # Special cases for specific variables
        if xnames[xi] == "Q_LOC"
            xrange = LinRange(max(0.5, x[xi] - 3.3 * xσ[xi]), x[xi] + 3.3 * xσ[xi], 100)
        elseif xnames[xi] == "DRMAJDX_LOC"
            xrange = LinRange(-0.8, 0.1, 100)
        elseif xnames[xi] == "P_PRIME_LOC"
            xrange = LinRange(-0.04, 0.005, 100)
        elseif xnames[xi] == "ZEFF"
            xrange = LinRange(max(1.0, x[xi] - 3.3 * xσ[xi]), x[xi] + 3.3 * xσ[xi], 100)
        elseif xnames[xi] in ["AS_3", "RMIN_LOC"]
            xrange = LinRange(max(0.0, x[xi] - 3.3 * xσ[xi]), x[xi] + 3.3 * xσ[xi], 100)
        end
        
        # Calculate sensitivity curves for both models
        tmp_tglf = f(ensemble, xrange, xi)[yi, :]
        tmp_gknn = f_gknn(ensemble, gknn, xrange, xi)[yi, :]

        # List of variables that need reduced tick frequency
        reduced_tick_vars = ["AS_3", "BETAE_log10", "DEBYE_log10", "DELTA_LOC", "DRMAJDX_LOC", 
                            "P_PRIME_LOC", "RMAJ_LOC", "RMIN_LOC", "S_DELTA_LOC", "S_ZETA_LOC", 
                            "XNUE_log10", "ZEFF", "ZETA_LOC"]
        
        # Determine tick positions
        if xnames[xi] in reduced_tick_vars
            # Create roughly half the default number of ticks by using fewer points
            n_ticks = 4
            tick_positions = range(minimum(xrange), maximum(xrange), length=n_ticks)
            
            # Format tick labels - special case for DEBYE_log10
            if xnames[xi] == "DEBYE_log10"
                tick_labels = [string(round(tick, sigdigits=2)) for tick in tick_positions]
            else
                tick_labels = [string(round(tick, sigdigits=1)) for tick in tick_positions]
            end
            
            p = plot(xrange, Measurements.value.(tmp_tglf), ribbon=Measurements.uncertainty.(tmp_tglf),
                    xlabel=xnames[xi], ylabel="", label="", linewidth=2, color=:orange,
                    fillalpha=0.3, xtickfontsize=6, ytickfontsize=6, xticks=(tick_positions, tick_labels))
        else
            p = plot(xrange, Measurements.value.(tmp_tglf), ribbon=Measurements.uncertainty.(tmp_tglf),
                    xlabel=xnames[xi], ylabel="", label="", linewidth=2, color=:orange,
                    fillalpha=0.3, xtickfontsize=6, ytickfontsize=6)
        end
        
        # Add GKNN curve
        plot!(p, xrange, Measurements.value.(tmp_gknn), ribbon=Measurements.uncertainty.(tmp_gknn),
              label="", linewidth=2, color=:green, fillalpha=0.3)
        
        # Add reference points
        scatter!(p, [x[xi]], [y_tglf[yi]], color=:orange, label="")
        scatter!(p, [x[xi]], [y_gknn[yi]], color=:green, label="")
        vline!(p, [xm[xi]], color=:black, linestyle=:dash, label="")
        hline!(p, [0], color=:black, linestyle=:solid, label="")
        
        # Special y-axis limits for specific variables with electron heat flux (Q_elec)
        if yi == 3  # yi=3 is Q_elec (electron heat flux)
            if xnames[xi] in ["BETAE_log10", "P_PRIME_LOC", "Q_LOC", "TAUS_2", "TAUS_3"]
                ylims!(p, 0, 5)
            elseif xnames[xi] == "RLNS_1"
                ylims!(p, -10, 15)
            end
        end

        push!(plots, p)
    end

    # Arrange all plots in a tile layout
    n_cols = 6
    n_rows = cld(length(plots), n_cols)
    plot_grid = plot(plots..., layout=(n_rows, n_cols), size=(1000, 800))

    display(plot_grid)
    savefig(plot_grid, "./plot_spot_check_gknn_$(split(filename,".")[1])_$(ynames[yi]).pdf")
    
    println("Completed plots for $(ynames[yi])")
end